# Basic RAG Evaluation

Evaluates the retrieval and generation quality of the news RAG pipeline using MediaTek's dataset from hugging Face: https://huggingface.co/datasets/MediaTek-Research/TCEval-v2

In [1]:
# Load the benchmark and save to the local file

from datasets import load_dataset
import json
from pathlib import Path

# "dev"=5 rows, "test"=3493 rows
ds = load_dataset("MediaTek-Research/TCEval-v2", "drcd", split="test")

records = [
    {
        "id":           row["id"],
        "question":     row["question"],
        "content":      row["paragraph"],
        "answer":       row["references"],
    }
    for row in ds
]

out = Path(f"./drcd.json")   # relative to notebooks/
out.parent.mkdir(parents=True, exist_ok=True)
with open(out, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"Saved {len(records)} records → {out}")

/home/hanyusu/newsLLM-RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating dev split: 100%|██████████| 5/5 [00:00<00:00, 700.92 examples/s]


Saved 3493 records → drcd.json


## Index into Qdrant

Run this from the **project root** before continuing:

```bash
uv run index.py \
  --data-dir   ./notebooks/evaluation/drcd.json \
  --collection TCEval-v2 \
  --db-path    ./notebooks/evaluation/drcd_articles.db
```

In [10]:
import json, os 
from dotenv import load_dotenv

print("CWD:", os.getcwd())
os.environ["ARTICLE_DB_PATH"] = "./drcd_articles.db"
load_dotenv("../../.env")

with open("./drcd.json", "r", encoding="utf-8") as f:
    records = json.load(f)

questions = [r["question"] for r in records]
paragraphs = [r["content"] for r in records]
answers = [r["answer"] for r in records]
doc_ids = [r["id"] for r in records]

print(f"Loaded {len(questions)} questions for evaluation")

CWD: /home/hanyusu/newsLLM-RAG/notebooks/evaluation
Loaded 3493 questions for evaluation


In [2]:
# Init Embedders
import sys 
sys.path.insert(0, "../..") # project root

from indexing import E5Embedder, BM25SparseEmbedder
dense_embedder = E5Embedder()
sparse_embedder = BM25SparseEmbedder()


/home/hanyusu/newsLLM-RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/hanyusu/newsLLM-RAG/.venv/lib/python3.13/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


## Run Retrieval 

No LLM, just search

In [ ]:
from generation import hybrid_search
from indexing import fetch_articles
from tqdm import tqdm

EVAL_COLLECTION = "TCEval-v2"
TOP_K = 10

retrieval_results = []
for question, doc_id, paragraph in tqdm(zip(questions, doc_ids, paragraphs), total=len(questions), desc="Retrieving"):
    dense_vec = dense_embedder.encode_query(question)
    sparse_vec = sparse_embedder.encode_query(question)
    
    retrieved = hybrid_search(EVAL_COLLECTION, dense_vec, sparse_vec, TOP_K)
    
    # Chunk-to-Parent Mapping
    source_ids = [rc.chunk.source_id for rc in retrieved]
    parent_docs = fetch_articles(source_ids) # full paragraph text
    
    retrieval_results.append({
        "question": question,
        "id": doc_id,
        "retrieved_ids": [rc.chunk.source_id for rc in retrieved],
        "paragraph":  paragraph,
        "retrieved_paragraphs": list(parent_docs.values())
    })
    
print(f"Done - {len(retrieval_results)} queries evaluated")

Retrieving: 100%|██████████| 3493/3493 [02:33<00:00, 22.82it/s]

Done - 3493 queries evaluated


## Save the retrieved results

In [58]:
import json
with open("./retrieval_results.json", "w", encoding="utf-8") as f:
    json.dump(retrieval_results, f, ensure_ascii=False, indent=2)
print(f"Saved {len(retrieval_results)} results → retrieval_results.json")

Saved 3493 results → retrieval_results.json


## Compute Evaluation Metrics

### Retrieval Metrics
**Recall@K** - Is the correct document in the top-K results. Binary per query: 1 if found, 0 if not. 

In [60]:
import math
def hit_at_k(retrieved_ids, gt_id):
    return gt_id in retrieved_ids

for r in retrieval_results:
    r["hit"] = hit_at_k(r["retrieved_ids"], r["id"])

n = len(retrieval_results)
print("--- [Retrieval Metrics] ---")
print(f"Recall@{TOP_K}:    {sum(r['hit'] for r in retrieval_results) / n:.2%}")

--- [Retrieval Metrics] ---
Recall@10:    96.59%
